# 🚀 TASK 6.6: FINAL LOCKED BENCHMARK TRÊN GOOGLE COLAB PRO (GPU T4 / L4 / A100)
### ĐO ĐẠC 2 CHIỀU: 18 TẬP TEST CLEAN & 6 BIẾN THỂ DEGRADED VỚI DỮ LIỆU DRIVE 5TB

> **Người thực hiện**: Thành viên C *(Training & Evaluation Lead)*  
> **Mục tiêu**: Nạp checkpoint rút gọn `fatformer_crossdomain_weights_only.pth` (2.31 GB) cùng cặp tham số đóng băng $(T=1.0811, T^*=0.3700)$, giải nén dữ liệu từ Google Drive 5TB (`test_benchmark_gans.tar`, `test_benchmark_diffusion.tar`, và `test_degraded.tar`) sang NVMe SSD Colab, thực thi kiểm chuẩn đo đạc 2 chiều (Clean 18 subsets + Degraded 6 variants) trên GPU và tính khoảng tin cậy 95% Bootstrap ($B=1.000$).  
> **Môi trường**: Google Colab GPU (Tesla T4 / L4 / A100 SXM4) + Google Drive 5TB.

## 🛠️ BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA
> Mount Google Drive, tự động phát hiện thư mục `Fatformer` hoặc `FatFormer_Hub`, clone/sync mã nguồn repo và bung công cụ đánh giá mới nhất.

In [ ]:
# ==============================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA
# ==============================================================================
from google.colab import drive
import os, sys, glob, shutil, zipfile, base64, zlib

print("[*] Đang kết nối Google Drive 5TB...")
drive.mount('/content/drive')

# Tự động phát hiện thư mục gốc Drive
drive_candidates = [
    "/content/drive/MyDrive/Fatformer",
    "/content/drive/MyDrive/FatFormer_Hub",
    "/content/drive/MyDrive"
]
DRIVE_ROOT = None
for dc in drive_candidates:
    if os.path.exists(os.path.join(dc, "datasets")):
        DRIVE_ROOT = dc
        break
    if os.path.exists(dc) and ("Fatformer" in dc or "FatFormer_Hub" in dc):
        DRIVE_ROOT = dc
        break

if not DRIVE_ROOT:
    DRIVE_ROOT = drive_candidates[0]
print(f"[✓] Đã chọn thư mục làm việc Drive: {DRIVE_ROOT}")

# Đồng bộ repo
REPO_DIR = "/content/fatformer-xla"
if not os.path.exists(REPO_DIR):
    print("[*] Đang clone repository từ GitHub...")
    !git clone https://github.com/LeeVHoangtk3/fatformer-xla.git {REPO_DIR}
else:
    print("[*] Thư mục repo đã tồn tại tại /content/fatformer-xla")

# Bung công cụ đánh giá mới nhất tự động (Offline Self-Contained)
os.makedirs(os.path.join(REPO_DIR, "tools"), exist_ok=True)

eval_tool_path = os.path.join(REPO_DIR, "tools", "eval_locked_benchmark.py")
b64_eval_data = "eNrVPV1v3Ehy7/oVnVn4zLFHo5mxZHsHNwfIkmzrLEteSV7cYTIgKA5nhqsZkktyZOt0AoLLD8gFdw95SIDkgkMS4B7ycE9eIHnYQx7zH5Rfkqrqb5LzYe9ughNgD9msrq6urq6urqomP2ObDzaZHw/DaNxl83y0+RRLNmq12sZzL38ep7Mg3fzZ0W6XvQqDGdubzL2IvZrEHjuL8S4as7157LMfsf04hIIwmLOX3pwdjycI75x72SV73Hxc39hkx17C8gD+pTFUy2KWznM2jqMuG3n5iJpy/TTOsmE888LIfReE40meuXE0vW4m+QRQ7GPNCw/+8xHZxJshHmrVJ9JEe9tdds56rN1sPW23G+z8Ady0mo+etFqIxIsmbBx6rAOVsGoeQ81hGETdDQZ/7aboSLvL2k9ZTmRnOdubBgDnPGUvdo8z9pC1W2w/HI3mWRhHdarZkTU7XfaYXQBGoDFg+8E49YbBkDlfJcHY/fpRq8H41Y66egJXF9N56mZtedFpsGH8LnLnCXLvPASqidKZByyE3wzZFwILhsE096CHu3t7rmprk26JZqv2JQwe8C8PI+DhNft85x57Fsd5BigT5jzrAc9arTpwJobnPnIIWezHOZvGIWB6GeMAss1N5Ik7nM9m1wAEmC5xwPPJHODGIVSc++wiFngkq6H+z+ZezodQ9uMyyNnXc4/tnX3Jrjz207OTYwDAajEJ4kY4S+I0Z3Emr7JrdZmHs0Bef5XFkbweT+MLeQ2NgEjJOy8dJ16aBRujNAaCr5MgY+LRGfxOg2NvFmSJ52sImB0SZD/08wY7CjP4/3wO0A22G1032EmSgxh4U0VtNJ8l18zLWJQoWuPUn1g3zSgikKhY2hzNI59jRIDnghR6Os/DadYcejDmkii4zgKgCC+OYhj/1KhwFaKESlhkeYazLeMgbw6P5KPDmTcONjbenJ789GDv3D09OcFJFGfNxMsnTe8iw19H3n8Vh5G6GYZpBGxzXHcUTgPXrTdYrdms1esb4YhZCCOQJJA9GEKqyaecvGuGURakuQOTwaxU30A0Ey/z8jx1EDjLh/EcOlxLAz+ORuF4nga1OkeWp9f8QmLmwE0D1AkirvR6NVJ6NT5/g/d+kOTsgH6AZxpN4mVZBQ1Bmq5PAwB/dxo2QPm4Z2+fnR2cn8Hg9OlpLUnjsRfVgJYsv54GhesO3lyEYyrmFfxr3wCDKcEvx95cXA2DIBl5l0FtY7CxsX/4/Pnbs8OTipbH8xC0DdaYDmdup9UyLl1/NJa3bXwiqkyhirvTcjtPqE26hefwz77H57zK0JtOA3yYRMMZ/l59PZSqlyjcPTpy208N8kw2PWSlDvAa+wcvTnf3D/bdL3dPD3ePzX5JTY2NSV2tr5/QtdDX+pIYLXQ2kSXwQkNne6eHb86BAmzjptBGl9WOYa346ZuDFyyD1fOL3qMWc3bTPBx5fs6evn9al5xQtJh18hQX4gvU8V/0dlo27BMbNoIF6YveEwUkOwEwL7x5loWgkZ9BGXOycDzzcD2o27CdhbAdA1ayAWD34XLzbcKehf78ApYGp9PZZps/Ye12B3/grl7buN3Y2BgGIzYKoyEokRQWl+A9qFmYII4PCjwEhRdkXVK9falt+7BsDQZ1xGIV8UkDWo75jBY6VV/NJpjMqIp8XYJ/MDuhxXmgCn0X9F6FFvTrJib5lCjOHKpUtzGnQT5PI45PPZh5uT8JED+uV038z/EbAOvP0yy8Cnrn6TywWhI1KnEXaRSw/daAoxBgx3EEWn7Dn4I+Yfu4gD/z/MsLKHVg4XkdD+fTQBAPy698RhbTFGwEXNELaz3YFGSSRezFm7dbskYTF2/EgsPqumEU5q7rZMF01MDV0SUCgqzXMTiVzZMgdepNBa47jxWbMD5XwC2gcw+uOkPnUYO1HzcY/MC4g+LogdWUeEPSq+1C5ZHPqx6FUeClTvsxe0AySP9bNMGCIwkHIXrnpUNB93uD1glgew4afTp3FG3O+3rdApg0r8LgnTNpZuEvAqcF6+KmQZYYEUGdM6mrcaG1GGVdrO2O+NUDIwrARvTZEE3azPMnDC9gYHwwWRlYer4ctmWDEebBTM4stG9wCpFZMxjAw4wbeV2w5uIpdtmbZoE5ZEg8oYCH9Ft4JBDgU3FpAyiLBCC0dQIDDFZJFjh9S9QNgNOAeIrapEFKpN5YBHoenwdRFoNkLQQ5hv+9KSKcgcnc67ea20/b2zvbj0G64HrnydPOkx26bD3ttFtPHg1sVJV/sOojps7jp487n+9sY/XO4/ajVmfnKV0/2Xny5Em7PTDIGhiy57rTIBLDZHBcSA08czTzrWrjIMdCPcJDU3DxEfBa1+3D84F6PPUugqkYy36N7moDUwdZw2provc4hmSlotEd4fRUw6MAAxCgbmEcCniopdkYsNFEaMYJdJYTRAYm6rfaoE5zDg3G2umLZ7V6CcV72U010A5gteEWG1zlTv0igP1pdafEoLxvcP6J1cwHIZ7ngQtrs5deu7MAlJSfue/CfOL6IG4XDiFANZrDFOStcFnlIoG2WbDgUTADZelBswFsoacx7KvUlpcDTNIgm8TToX4sNsG0YuqpTk8HSrPQ70u9qY5g64Z6BvbUsD9LJlAEFULc28YsGs9R+Z8/aFrVQUxQPgX5ddaDtksSfGOxujbzYPvver4PRkOrCXuAWgyDC6afVZYGnllgo0CT1YL25ni9I24Sca0q3fJZk8FYBEOXDwOwSVxsmTwmQDC0yRx43sziUT7z3jtWVdixh7Oe1PBEjKxBv/0uLFeDpp/MYY2jHaJY4ASf3ChBWeM3BTDRfjBEbI6B+yc9PdL1ppfhltYJo1xUyRMXWYZLX9LM5jPHEUhgROrguHHMtrGMUxQVahWhhEXBxwIpku1sMWQLrw69rddxfW0h0yU5SHsVOe0KctqKHLtWEUozXJNDNSQ5eFNBjpI5mhw78NBRnXqoEHL0hjRiC4ISRb4mqS5aNWYAllY0/xk7Pdnb3H27hzNr943oKqx72FOiWRRFwZiKkDg5wQQgEM7A0hUw1jwD6W8wD2WKzwAl+7YKjlPYnHLeArWgYHJDvgwV50WXHApmRX7tTsPLwKG6NkyfygYSIfh6AmKFgVNXgC64WAl7TJULI6tXnrkefVUJFJlgAwwcv9hEOdoCX1zLZAMO2ByKHQlN7CoOB19BM9+t4MhmFUvwCcx+QTMzhaCvEek+5EnGsfrzGXakWF+JMonzSuCWwUfYNJCjh6hIUHk5+PMQ0WgwEgZDdGUdYF8VLXU1f4BrFfJbUuSWEqd1xVElhpVT0Owc0CgzQQ2Fz+FkgQlkKH4OpKauAcQXA/4c2ToNE4emCE0O6lbdgk/K4EkV9G1xuZceVdcP/19W+IZQG4qQLuyDqXZLPsyCYChLtztFk4B8m31CKu2DgoFALuWV3uQKZ7JtKABjuX+2iQQ5+J/U+JmH7lhajw1TQms/+fzHbKdsWoiV/QJlAFH0B39eKz06L1x0XnD9aQylYcljmQvGO1cTgo/A7dDHGoI/sDWGbU3PuE+DZAr+7YJ3IZtfqD7oDvVlGwMLsFLjaVBTLXGDRWNfbK9oLbnCZLEo0DZLld1SAtWSIhbyRYbLhknPcpul0EjboseuWgLV6l7TU2m5aHpmBYNFWiuk75SCVkJCU6DpJbCHGjozbs/YkwNpwzWOF3CS/BBE/R3KAilA/qgP4+QA8s4OLaFaJgeq0gSidtW1Pn9SUWvVMgIa1P18B/Swk8ZzoJ+TBZsvMKdUCbaJRXVLG1/MwynOV/8Spu0shjCZA/4w/zKByEVOW8guI2WHmt28HwZXMIOkRuZ3KzwhoM4gLAp7IghczTAkCcrxCu9lsHPoyXgnKwY2m8amqbytTlLkXq3/YMBUC5bXjqtXcOsOtWOuicGXok4s+PosF1wzjx3eUal/MDiUpX6TGKdCZJynVMbtZ7xy0UYChhTiZ44WQdFor7Z3dPim+2V4vnm01d6u6bVWDUFPXemHFqm6eJ4FbpbOSI3ZpWMP/caFBwDqQhzIRcXdbsEGnp7UNzSTR8TlfVzRLpHVQ3SrX1IYNwUT0oqIM+fs9DV7tPkqSCPkz0P2ghqta9YTb4AtBs8czS/BZxjzglCCD29Y8ijbIPWieBiUR+SjleC0QoP8ddlNAcetISAl/wvEhPJAuT1gKg+LJDRAOyU4uTx0mvRqsIRB9MMU7R6fHxsFR8s5qPuDNI3T76NFyzcOpmwEaCKQPMKGa7Sf14mfNc54Kq9RABKvqkmg375Vw1j8BNCNc9l/0h1gu5dNDKHlGXp1HKwGDnSQAdpksct6l13Ran7ZgAvZMne8OfVbY5LiFKJGqfu8ZRe7ILuDLm4/L/J1FkIkJhq7PmjDXBhLsLxQnDLF5b4mIS6D6wwGqT8w1qd5FLxPAh8t/8UIDKAqHEoG/+fvfzPgeR4TdESDdw7SSl7z1ns3FqG3DfZWYe3dFMmQ0mlvVFVDfzFge9gC5Al0US2SDYqaz7s2hT8HL1VJ7jFJA8As7TyhSe6FOHWtucvHw1CO6mEzuPKmjhVZ4WqRLz74dE4DyP30Lh9wXb/LVLSFa6Jlbnjh261YlvhCjpEeF808Zec/FlsAUJ3v4vQySDP5SNj/1cvZ9+Na5LuHooOxYW9xCndVTsiz+TWbogsyTwNkb07ZOJM4AVUwR5cJWBHsKjAzVWCd1flAJZ8kZR8Qn8sbBkkQOTcoVFO8V9teMabSPW3GaQi7NhZ68qIutoEedyzoTA1nKLM39Cj29CXM+sl8NJoGfNo3zBHtGddCbtG8VJsbueex7PW+sM7zFmpbSJ9p4n9CkFGFySSU2MVUIsdYa1CLgYP7GrUY70nZ+/++OFnkn4Ea9iFNb56DJs9yAdvEXUCDBZF3AZusnlmKZnLNnw89SCcpO+dhRZfTFIJvtj5XvJDGL6aB8L0WN0/r5QpySyZqXAu+BhBDyyCtymIZ+JpybmfDmqTZzjsJRY6mgO8XYfeC8RM9RrRE2GKm8BlbLAuhKC9ilOBVKLnW4cEHwLdmVMLuWMMkSltVhqroGdcGgFQVPXVlGF5Sc3ISGgvbkwMgtCvsxWE/O78gxYpTMHPEHWQiCSsedk9yy6tVX8HtQdqqSuMqPfTFHOY56heKrNJ6gYHvGdiDvEXmtPj+EdRRmzZroCn8mDL/xiEsMhgvAcUEGksZ+uTBwkCWMUtpy22UbahYHC5bOZiZmUpPwT+nxtuFxRh1k1Ozblr8yoDmtNVwJ4kA1k2brgh4oN0PvHHiyfRiSpPeIqdrZEkA240sCcoN0wPS0Kgq0ybCDIAcQlKY4UhGCtvFBnNhhSDmhJTo8M6bXlbWkLVGEUJSlW5loBbaH0Vgab2jhAOY69x+c2rNJKK0peZXifwN+MW74CKpVELaY1tkAid+FNUX1sFIGXDXdtxX/WmhkcrpxgiGwkKcYK4VhWsh+nW7uMVy/LXUDyWLazTVvtVbGWPOsZ9AXAJNbzQnNe112CmpYIAo+Yk1V/XKHI15UAC9Wqf0Y/gHKcPCm45Q+xutboHrXxvrVDiULiFA2ORFBkGgKGCcCkQ2CLWxPEhMwpljYNId0ZiMzll6CBYMitobZNWNRkAvg0qmVcai/KHVvMmfpjAOHFm1tOmXDyxHi6F/HhqDLZQrZl/LZZ9SZXFqy7TZ5m46ns+CKH9DTxyMcqRhwrdk9g75eYjpq0fkf2HPIN1xMvPSSyMpXNj4n4FBrkz2HzHpoTDab0JGD26aqWGntrmpbXqYmGgk9ITjZuTNp3mvtr97vgt5flsaLttaL8tc0LSoXemcqG6WHBub7W1ApPom+k/rakpbV0QOPhLQoZlixvbSZo2FVTYsXPOyaZnqPgmmSa92rCP2Q5UyL1ta3kO1SC9oSMQXZEM8+g8rNSbZL2zrM5minC3vJST9ocKsZm5NNnou1t9x7JOZX87SX97FociR/6S2KvL6l7cmpr4LA+iqlvnaKNsG8dRt77RaizOLOFFnZE2AkRfidmcGP8IUGaqTBs/BtN48uEJPNrhSwVQE6wUxg06eemw0n07razLpykshzzPPqhkF8braKnLvA9B9no2mE9UUFyPIOEYDC4qoP3PMMLlWgnMKzgAwtkEvnMxzsFqX0RwTBBoFy7UCGJjZ8t7r/VflGLUfL61tbMsqq7eW19YO8urGMaa2ch7RphPqez5XzRlsCQIX/JiBku094S1BRcFT9Oy8zlUCgju0akbTbg3tAbFzgVtMGPOuvHCKGzxHOMeED4+aET5k0Rr9kJ9Up76QH7xXgxjC549Nt23t+e7585PT1wen/KjS88Pj3SN2dLL36mCfPTs43nv5evf0FUjQm5Oz882XJ3uss7l/+PrgGDPCAfLLg9PD54d7u5ieDQp59+yVuTpJn9N5+bQRTD50r8JhI62f6eDRDRLeNL0oze3RLfslO5falZ9JEnDKmYJQpWZRlV+EXQHM2Y649vjIybxch9z79a5qXAqBjVEz0PApqT0mv3WMluoyUWWfQhqh6V6DHQ8uiMKyyOLpFbrvLhPcW1XlcevNCzWgMRkx9++wcq/AMtveMqtz9+4Fsmm2Xay/demNx9NgK4xAoWw9ePBxJKh8a8kSYNKaLCkHPyyjwiiGECSoCDTtthZASCYshRwUHJ4VQTNraBt2t2SwzPB4FcRPCVDv+/qTRuPLw4O3eEhvf/f4JXtxuIt2wPnuG3Z+cHbO9o4Odo9/kJbFPPrLqAbGc6U+Asruvvkrou1Pvz6Bf3cf/nGPqLv78AeTPkM7OeJUoTqqUl82bX20clw8TZQOM/fiWhgUqFJupXeXw8jc68KoUMZYnBvFaAzJ0kL0yQbSuz27keUhyyEsKqg50XApn6pMMGX9aq4XINBtObr17UAmbuyhp7izt0/8FE5xVHOnDz8Dy4ci/wq72xr1R/TtBmrdujfhrfQLyH1vyO6xzm2l8yHUiRvbLXsjPlgW1pAxvK/R6YQBjTKnLigaSYGVG3tcbj+JV+g6rHDe2KgbCGb3QzjTbReOQFbhIdEnXfqV1mJC9CZIrT4Ks5Si2oMHNU7X4gMzBYJtYpMy3GCjolblkZsy+/Rpm42Npe0uZhJ0AzlU7dzkjjTDmdArSFa1W0fg7A8+ZoZgJSP9Jc7BXcCrjDDlAmmEVBZ0ZVzVaeAosLkAaRMjYdCHuimbgEU8JuourQjpIkQiZiqTy6/Qc9QaLAhDnst4YE47VPzvptSTW+HWhUl1Q44bk7D67ZYx/2C+WU9va0ZGDpm6JS6VXHpSMS4LX2J55MUFXzNSgIHL+TWQmtEJeNNwL+pPWxVULgg4U0q+1jUUxicp2O9bya6haA1n9mc6kMgPxJt9yGDHYswQXoxpqMUwmn5SDqatwTucwvKY1CIBB6w41Sm8XjypqOovOLGo8wHWjaUAIcui1AofGoE9+t/e6RNsT9FlP+RmYc+MUutsIBXnJKVuBDstODPSSYBGQaMo3Craakh54bCXEakq7s8KkCpkZe/PFJAx84vyIh3mmvn1CthClNEYHQP1GPLbEhfP1cNI1dA+rIlgH8qZecKZb6b39ZFoI6kElst36syx2hsQbphfuo3CSRrhoepic4VHOAxiDYLnqDZN+huLD/UIyewbhYVTdGayt4JWZYPFp30UsCorAvPkbwWHtyWQxIJISgBi4rhZgLh4+qEow/RD41hRUVOSlpHDLUalnEDDMBzZv9GD0v08ux0w1I7d9k6Gu/7XyDp8twesR5LU+4qf9wfdZmd0ew8AawVtOKqdAh/NWpKvRqXnwD0TRHJzKV44vmLWAc4a4LtvrGeJeuTcCM7RfVavGScYwVnuu3gOE+Jn70BhcYbQmtAQvk5+Vy8d6MYKlaejj2VIYfEBNHtS8OYWzwpFxNLpgYZS2rcKBzysSUsXULtsxqjDB8SMfmpNHRvNoHj+tXxwwsSjJ9UqNKWjFSYaPd1WoSkcvlAYaCKurJxU101WV62as2JQzEelQbHnM12Cp5omLq38qa5gmxIgiICby9KAkldIdXMjAl9b8SkotG4fiGQOAOeVkCq0KdScUWQ2RNPw+xq8DS7cnMncVF0fSbLqKxobZrsSiXJQlDGJAzwWMqtjgOOEwyh0J+B9BZOpZjk25EmgBWk1xTXXTIUp2W8LU2xKLVUl3JSW7Oq2VibfcFA6IOGHZgpO6aTQMi40FlNdnZGz2M5ZZOMYWdeaOGF+6QIjg4dTBEn4ZIYr41Z0lUzaUjY/5JzQISo6QlU18Px8Q1m0bP2Hm/Oy9NEYqFQ921u3aXrr7Oo4a6pdMtxVh9LKX/1Ff129EqN1VUBUvSTjOc8q2OKKWQ2kls5aFfk4aVeSr2bzEvIR0brkc9gV5Aug5eSLwVvZA6Eulg6AlJWKTjh4Qm3vsAtGli234LbhIA1WfNIWT+A9Mwusn0Xtr2CMhivyRgjtpuXz/aE86B3Dg/6YPTs8OGbnLw+YfDkSc77cPWdHP6//IEToWHxxIw5r9AyzE2F3LELQ2rsNmmoOUQrcvaqYOsYuIJFZxq6MEH/VgYoyFBxthcW44CZXru6qFksO76qQQEVYoGOFBZDldx/+A5l+982veHBA8x6Avvk79gzqHbMvIXoAI/HtP+DrFg+OznfZ/unJm3qt1JAdKxA9sHsso/pkXOB6bNvM8rGbx9RnfLtg1QuzlrzTowJF/6qJRxgSx3AaVpPVzJJpCD1p1GihVfUM3yru+eH9VuvEN2xpWBLjsAViw/KlA2l4trFBF7QpB+oDsOtxaQ2cYnfrJW64mCoFlFa9FIyWSYlYN1Gv8B6OQMz6N4IafA3l7Ra5LksEwMbxT7+ms0zh3Td/PcOY9d2Hf4MIQnj34T8x9wJKu+z+jWysOU/o5Uu392FvJum9FeeZSp0Rzis9J62DOHpoys7Dz2Tk2n4XkJ1pZzqHF7zeTr96jr8eblAZihHeEhtrtS/f6NZyT+ZibyZ2WzH09pPcmhWuzU6rOqQxWMP/TwO5IL5jifsiqVsY7BGYF6TDrgr4fETQp0AmD/woYteL/qwbAaqOAq0TCbKZvSga9Inc1A4ijB28mnz7R4wNfPsHdPbfffg9BDkm//XvbHb3ze99Og5qznFjisPUzu8+/BMcihLNwWEoWFv+Bt+/2qxVc6PsYP6EgMGSSKOgpCLE+Imhxo+RwPWkUJH4sTHHj5W8xdK3rgSuF5P8hNjk6hjlOjQtf2eaGdykmb8kd3I5JkwT7213FsIsVRLYxeUcrlgp7JCpBMJuS2/PoCraJt6+h8JnIJWRzoo4Op2E+A7RnJVRHTu6UxHZWSPC87GRno+P+FRFfpTNUQ26fvxn7TjQYnH6HmI3asZVxG+Ud1NYfWBSKOtxKSAZdQIaLxdArwoPrREm+qRwUXXYaEnAqDJwtDhkVBk6Whw0KgaPKsNGhfBRVeDoowNI5UBSlVZR8SQzlqSnp72ztoA3bIOc3uUzo/QikZZdNOnNZstaaVzl0rb031KnuEX2J6AqOsdt0qhfvXVDKbwrEEbAdkS/lEexTOzHIR+ayIcrkMcfidxk0qBeFoh0Mb6qYNAqdKPF6KqCQqvQwcRajLAQI1qJK1mCKlmGqWIWwbcV3GEa08k+jnaz7KwuVyy7r/pSQw/+79U5RoS4tuM6h88KeotOdQ2KAZk1hqtq2G9045XiVZUMrc1rpKtqGIqb1xitqsF1NweWUrYMPNHQyUo2KfHQjFJFa2r0QvBfDnz3KYb+RbhMBPw5P6Uv+79/C0W6ue5DKq/yUmtvNWpdqDU2Ef2S4gKIyy4V+QKpXSpSBEZG6ZJ3daCzinasmMwGB3ty/qqNwiEs2IXG9A2Qofw2C/8ySwdONQi4uvJCff9+8J+9BQ/388OjA/DNsWe7J2wP/p2fHL9gL0/e/CBtwq5rBnyEHZHwPOqjS2DQo0fSjS/FxlIQAJ+loW+nyM+m8HRg+GiKGYXLriodPSZ28FvBsUx8nyMkWPtwAEKcy3Rhs5iG75uAQ4wnzmZ4Kwu89okHLPDVMECZHdJdHUorolFvIrMrrA5qrYEIK6wVXloDl6gjBmAS4PsuMsUJaYE0lA3csE3dhmnEmu8tbhhKTLyhmL+buGHbhwP9Rg56/bUaYoB7V8N3ZRS+H4JfijHYNWq+S2H7hk50LgpWF/AcNgYs7IRxWhSLvCkcyS+izfA1Qf3JgDv1J4hhSUuGE5vSdPc99t4Ua8pg5ceruISnXleF5LKrW60FOoUJoT59hB8SivCQViwloWgLd423vY4/ZtaIczEKX2nOmKIypHf22s4ntcI3Cmt4g60UqEVpQp8iXBtld5OWNMGUNeVsqawBqrKkWdK2eGyWSxyFTOClYDV4I0NZ8irbtRPEl0ieEiUSPsEOQ/QeCdHDEID1wSqebA4vKBIHwGgS4XepPlLAtFbmOJqIQ/QELyHucI1v4LEsyRqk2Xno+gFbpJDZBu6gS/wUizx6v+hs/o/waCQct40y/u2pA+5RovyfAkaAgteSzdDwoVfiwKCM6LU4tXs/37w327w3ZPdedu+97t6DnKNCZeP4flfx5QJcVvQRKeu8WZ175MzDhdxjcry1W6JJO3Rced4eIkRdttzdU1OOnapa5cQY/i0clbiDb1MNffy6jKxjJMosmrSQBQMZBugP0SuqzDgockv6uOg4HgaP7FR+jBWZsdAF0WrONruuTnAzjg3y54Ee+ZI0ib1N6YEQQrGgUy6ttcQ3FkDrbzl1i6v/oipicbYriEKrym1lBicSZ5kxlZxQqmkNZhgbPZ71WShcSEZR/2lKlo+kymcVyYlafRtqZ00VjqBNkIjEMZULvLcG3NPREA5842d1QCngxPIyPwzNdxBWa1WpEEk7Sq3I9alBn6FTt5v4zQl0PoX8HRKqkk/fmfTxO5PyO4LrncN8dvfhd2DEn99981u05e+++d0b9gpyMH71GlJf3t59+Ndj9urlt7/ZZWcAgUUAtfcWEi72mHWkfNGhcLvRj05o+o65TOVUpD+D7CaVT1OaIIuyaTars2lGtZv7z4wo5/3uj9utW3Zz//W3f8Qg5z9DE9ciDgrPHtEzSt9z7tUVMG58rQK53xZlHSzDfTAOjQLCLbBV8N+/RRH6PX6PE1qe8ZrlvJwlXSFbV/fh7pu/hTNf6d2Hf4n4Z05B7KEnc9mTpemGgATZvSKrrwqqUtqgL4sBjTMESxEaBwkU3H3werbAj0G9ra/NNJGMA3FYSuwoCVJ18EyxGqAEm4FtpgmOpD0S5dJhZpILxdIrVig2XF8mv7Jq9mTV3ED0ypcDDx6arpVF/CglfOFHKl3yH7ku+cZdl95U4IoEL/5up43/Bainee8="
with open(eval_tool_path, "w", encoding="utf-8") as f:
    f.write(zlib.decompress(base64.b64decode(b64_eval_data.encode("ascii"))).decode("utf-8"))
print(f"[✓] Đã nạp script kiểm chuẩn 2 chiều: {eval_tool_path} ({os.path.getsize(eval_tool_path):,} bytes)")

plot_tool_path = os.path.join(REPO_DIR, "tools", "plot_three_way_benchmark.py")
b64_plot_data = "eNqNWMtu48gV3esrLtgwQE5LHJJ6tCRAC1vuTidpP2JrJgEcgSiRJZExVUWQRbc0g1n1YpbJYJBFkCwyCbJIgCyCLALYiyw8P6J8SW4VH6JkudG0JJJV91W3bp1T5RfQ+qwFHvdDthhCJuatvmxpaJrWeEPEG54sadL61bvjIXxJ4SSkGZxy/IQwDuRLG04oA30SJJS2fknWMObLmCREhHdUdnnBkiS3BkqQ9BZ6Zs9oXHNICQtAJJkHfkjZsGGbcEJSGoWMwk+4B+MvL6/AsZwO6Ge20XBMOOMQhKj0BkVEhmKXEWFwXNjtmjZ8DmeO0WhvRU8JnKF1mHCUPJVP+lkY0VRwVO9J+Q68IVE0I96toQbcCJcxTwTwtHxK19Ujy5bxGkgKLC6blkTEERdRODPjtXyS3XEkGo3Lq4ufvR5P3KuLiwmM0KAZExGYZJbKu16+/4aHrHrxw4SRJdVdd45huq7RBM00NcNoXHwxccdvj68m7uXx5G3NntKv+0KV0+PJ8fXriYaPEV+k8i4wRy7m3hVymtz3ZO16aprClDMzZgvNwLHPISApESLRcdRmKnyeCVROqMfZPFxkCdWMYQPw2vabtV4dZ1vV0UhTdSSNNnw6xzRhlIUqZgdV1xE1s5TqWkrJjCesdWe5/db7IBR0kYS+BhjM832sZoXckTAis4gCjVIKGvojWSTQt/SGgTVBJyu7CWTlGJg5pZnN5GSlOjY7TSmUhl/RkW73mlihXcy7H4ejtmXhAKSVF4D1eZ0rgT3EUk2w+oksj6KUu+AFoXzzUCLioVLzCEbMk5Cm6PdGNclLG0cUC/KMeAmH4/H410yfbB5+zxYggs3DB0Nr7ou+Ibe0kDwPNvf/ZLhqNg/fyqKez7M05OyA0hUlUaF0srn/KwfBH//MYLl5+C0wyg9pXIxbx19IhSs5pgX6+GMIcfD4Fwbi8Qcv2NH58Xebhz/ABCdjjCGt4fX4NWqO0YcayP3fYvDyl83Dd6JUneYZXdpu6vEkz0zXMi2rCfmvbeX3rm32cTo6fdN+NVVzgA4g9UhEfVihFPgUEGK8IFMOCf76HI5y807NfH9gOmip75gOTvagZ7bR/KBr9nH2u2a3N81VOjWVQS6Lmm0UGnTMQU9p9rtYMOZgUIxihbIsNhHt2ILqEWX6dsqNvADfh74IUMwynW6uhAtGpDY2YVWaM5LoK2jlYs1tWpplC5Y2jUbaFg8rmJTAiGvb4xFPRtqLvif/sIH6C1o2dtVVrAXl2Kk7bm4Tte+wQNcKbfdgtua4bfedmb/n2Op37YFXd9zeGfHLasSdZwLY4R6J1PrL0zAVYRTByzFnaT0E2uvZlr0XwgzXr2Vp1RL+KcOKJkg6IX6pXEJY47hcc5zgSR6mBJebfIqaRcaKe3s6rIq/FJfSqnPbJS854bLZXFDhBjRcBEI3diRkJghjXGC96HPt62Bo2vNvjmrr68m1Wo/0yuhKNzCH1atKHTZ9LuEsMD5qRdCVGOm4BNofk5NSHueJn440Pp+nVECMXCPSj4UYkJHmUSZognNxhy8zLgRf4suco6rE2FdmN397r9IiRSK/nCSZFfTkilBEyA0SYb5T+4zN/X8zmCSbh395iHkK+nDf8fgPNtyuh7u03BLgE9bLz4PHfxO4RmSt+7edA+6bEBMfu4ydKFYi9G5TfXWgVdVoWlvtNQ+DZ0ZYN7JWBnQNcf8/8A7HA/qRsROm/Qk2wqWcRtvubjsiuqDM1yPuIQfHMU0gonNRtzzA5wR3GSSKcbosc1AtEGfLcc4QrohfsdxMbvTkRg1mGW7OWIDbPCYBN8q8wrWDG4Elv6N6GYsEGiRWk/i+W/BtQbf4iXlEktEkyWjhPZHeXMznx6myzj8V91X0+CyjPe04vqy3Kaq8RBCiyR1uXDnbUZDbGx+3NgiCdgtJaEtl8vccA5bIvx1BkQG2iHIyYbgs5xEnQj834DNw8IukEYcKRZiCEMUf58a0rvlyVDzdDO1pOUfnCIkkwnmEJUUk81JkdbBaSId5Gt2l/YRRCybtOorxSn7VbaQdy+z07VcyKmyeliacA6yZ0yVSoNOvOLQwYXW7vX0TnQMsmtOnvDkVp5YmnMGgMrEdCaZA3lUCqtjyRqfe2CkaO9tUyYpUNZfnsKkMIbkgUiiwHNlymcpXtY0cacNDXPqEfRXpGs96cHY99Hc8tFqHWHOXb3XJrc/b79TtO/v2DzFiYX4HD0GvTj01X3jmiPZ8PbFWgoaEykpRIXZABXFzotDz6kYuMg7I4CEHWQvXmN6yd/sLvC2LvmVPD/QXyLtdbZ+MvLmRde7kxsH67eC3h99+vu2cHpAs3N1ojnUkD1Kd/NbLb/38hrpH2nSbrp66sEdF1d8zW8K21d1LT056J0hvHzKQ3Pd9AcInhMMvsscfBJz/+AFR953iwWuSwVvJhRmidLa5/zv7VJ6rRfSULxIpj5KzGV+5grsET/A4Kt022xi0aRufwCXyjCWkHTciazwkFqSAh9YlQjVWQPrkzLt7wkUndIW7PZffFixRHR3JHUVa2ZOvDmxF3CEGTXHnooLQttpexNOSouIEtzO497r535++n8r/FawyUlIdx08oj3TqHxwzKiF6CF/vOv1GnW/xoOq6cgiuC6MRaK4rT7uuq+Vbwvzo2/g/FeciVw=="
with open(plot_tool_path, "w", encoding="utf-8") as f:
    f.write(zlib.decompress(base64.b64decode(b64_plot_data.encode("ascii"))).decode("utf-8"))
print(f"[✓] Đã nạp script vẽ biểu đồ: {plot_tool_path} ({os.path.getsize(plot_tool_path):,} bytes)")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Cài đặt các thư viện cần thiết
!pip install -q ftfy regex tqdm tabulate matplotlib seaborn pandas
print("[✓] Hoàn tất Bước 1: Môi trường sẵn sàng trên Colab!")



## 🔍 BƯỚC 2: KIỂM TRA GPU & ĐỊNH VỊ CHECKPOINT M4, CLIP BACKBONE
> Xác thực GPU phần cứng, nạp checkpoint `fatformer_crossdomain_weights_only.pth` và `ViT-L-14.pt`.

In [ ]:
# ==============================================================================
# BƯỚC 2: KIỂM TRA GPU & ĐỊNH VỊ CHECKPOINT VÀ CLIP
# ==============================================================================
import torch

assert torch.cuda.is_available(), 'LỖI: Vui lòng bật GPU trong Runtime -> Change runtime type -> T4/L4/A100 GPU!'
gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'[✓] GPU khả dụng: {gpu_name} ({gpu_vram:.2f} GB VRAM) - Sẵn sàng cho Locked Benchmark!')

def find_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    return None

# 1. Tìm checkpoint M4
ckpt_path = find_file([
    f'{DRIVE_ROOT}/**/fatformer_crossdomain_weights_only.pth',
    f'{DRIVE_ROOT}/**/crossdomain_model_best_m4.pth',
    '/content/fatformer_crossdomain_weights_only.pth',
    '/content/drive/MyDrive/**/fatformer_crossdomain_weights_only.pth'
])

if not ckpt_path:
    print('[!] CẢNH BÁO: Chưa tìm thấy fatformer_crossdomain_weights_only.pth trên Google Drive!')
    print('    Hãy tải file fatformer_crossdomain_weights_only.pth lên Drive hoặc kéo thả vào thư mục /content/ trên Colab.')
else:
    print(f'[1] Checkpoint Khóa Sổ: {ckpt_path} ({os.path.getsize(ckpt_path)/(1024**3):.2f} GB)')

# 2. Tìm CLIP ViT-L/14 Backbone
clip_path = find_file([
    f'{DRIVE_ROOT}/**/ViT-L-14.pt',
    '/content/**/ViT-L-14.pt',
    '/content/drive/MyDrive/**/ViT-L-14.pt'
])
print(f'[2] CLIP ViT-L/14 Backbone: {clip_path}')



## ⚡ BƯỚC 3: GIẢI NÉN DỮ LIỆU TỪ GOOGLE DRIVE SANG SSD NVME COLAB
> Trích xuất dữ liệu chống nghẽn I/O từ Drive sang SSD NVMe Colab:
> - **Clean (18 subsets)**: `/content/dataset_local/test_clean`
> - **Degraded (6 variants)**: `/content/dataset_local/test_degraded`

In [ ]:
# ==============================================================================
# BƯỚC 3: TRÍCH XUẤT DỮ LIỆU TỪ GOOGLE DRIVE SANG SSD NVME MÁY ẢO COLAB
# ==============================================================================
import tarfile

LOCAL_CLEAN_DIR = "/content/dataset_local/test_clean"
LOCAL_DEG_DIR = "/content/dataset_local/test_degraded"
os.makedirs(LOCAL_CLEAN_DIR, exist_ok=True)
os.makedirs("/content/dataset_local", exist_ok=True)

# 1. Tìm các tệp .tar trên Google Drive
tar_gans = find_file([
    f'{DRIVE_ROOT}/datasets/test_benchmark_gans.tar',
    f'{DRIVE_ROOT}/**/test_benchmark_gans.tar',
    '/content/drive/MyDrive/**/test_benchmark_gans.tar'
])

tar_diff = find_file([
    f'{DRIVE_ROOT}/datasets/test_benchmark_diffusion.tar',
    f'{DRIVE_ROOT}/**/test_benchmark_diffusion.tar',
    '/content/drive/MyDrive/**/test_benchmark_diffusion.tar'
])

tar_deg = find_file([
    f'{DRIVE_ROOT}/datasets/test_degraded.tar',
    f'{DRIVE_ROOT}/**/test_degraded.tar',
    '/content/drive/MyDrive/**/test_degraded.tar',
    '/content/test_degraded.tar'
])

print(f"[*] Tệp GANs tar: {tar_gans}")
print(f"[*] Tệp Diffusion tar: {tar_diff}")
print(f"[*] Tệp Degraded tar: {tar_deg}")

# 2. Giải nén sang SSD NVMe máy ảo Colab
def extract_tar_safely(tar_path, target_dir):
    if not tar_path or not os.path.exists(tar_path):
        print(f"[!] Bỏ qua: Không tìm thấy {tar_path}")
        return
    print(f"[*] Đang sao chép và giải nén {os.path.basename(tar_path)} sang {target_dir}...")
    with tarfile.open(tar_path, 'r') as tf:
        tf.extractall(target_dir)
    print(f"[✓] Đã giải nén hoàn tất: {os.path.basename(tar_path)}")

if tar_gans:
    extract_tar_safely(tar_gans, LOCAL_CLEAN_DIR)
if tar_diff:
    extract_tar_safely(tar_diff, LOCAL_CLEAN_DIR)
if tar_deg:
    extract_tar_safely(tar_deg, "/content/dataset_local")

# 3. Kiểm tra số lượng tập test đã sẵn sàng
clean_subsets = [f for f in os.listdir(LOCAL_CLEAN_DIR) if os.path.isdir(os.path.join(LOCAL_CLEAN_DIR, f))] if os.path.exists(LOCAL_CLEAN_DIR) else []
deg_variants = [f for f in os.listdir(LOCAL_DEG_DIR) if os.path.isdir(os.path.join(LOCAL_DEG_DIR, f))] if os.path.exists(LOCAL_DEG_DIR) else []

print(f"\n[✓] Clean Subsets ({len(clean_subsets)}): {clean_subsets}")
print(f"[✓] Degraded Variants ({len(deg_variants)}): {deg_variants}")



## 🚀 BƯỚC 4: THỰC THI KIỂM CHUẨN KHÓA SỔ 2 CHIỀU TRÊN GPU (LOCKED BENCHMARK)
> Cố định $T=1.0811, T^*=0.3700$, chạy suy luận trên toàn bộ ảnh test Clean & 6 biến thể Degraded với GPU, tính toán ma trận sụt giảm $\Delta_{\text{drop}}$ và khoảng tin cậy 95% Bootstrap ($B=1.000$).

In [ ]:
# ==============================================================================
# BƯỚC 4: CHẠY KIỂM CHUẨN KHÓA SỔ 2 CHIỀU TRÊN GPU (LOCKED BENCHMARK)
# ==============================================================================
import subprocess

OUTPUT_LOGS = "/content/output_task_6.6"
os.makedirs(OUTPUT_LOGS, exist_ok=True)

cmd = [
    sys.executable, "-u", f"{REPO_DIR}/tools/eval_locked_benchmark.py",
    "--checkpoint", ckpt_path or "",
    "--clip_path", clip_path or "ViT-L-14.pt",
    "--temperature", "1.0811",
    "--threshold", "0.3700",
    "--output_dir", OUTPUT_LOGS,
    "--batch_size", "32",
    "--num_workers", "2",
    "--n_bootstrap", "1000",
    "--device", "cuda"
]

has_clean = os.path.exists(LOCAL_CLEAN_DIR) and len(os.listdir(LOCAL_CLEAN_DIR)) > 0
has_deg = os.path.exists(LOCAL_DEG_DIR) and len(os.listdir(LOCAL_DEG_DIR)) > 0

if has_clean:
    cmd.extend(["--test_root", LOCAL_CLEAN_DIR])
if has_deg:
    cmd.extend(["--degraded_root", LOCAL_DEG_DIR, "--samples_per_degraded_subset", "500"])

if not has_clean and not has_deg:
    print("[!] CẢNH BÁO: Không có ảnh test thực tế -> Chạy chế độ Dummy")
    cmd.append("--test_dummy")

print("=" * 80)
print(f"[*] Lệnh thực thi: {' '.join(cmd)}")
print("=" * 80)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"Tiến trình kết thúc với mã lỗi: {process.returncode}")
else:
    print("\n[✓] HOÀN TẤT VÒNG ĐO ĐẠC KIỂM CHUẨN CHUNG CUỘC 2 CHIỀU!")



## 📊 BƯỚC 5: TỔNG HỢP KẾT QUẢ, VẼ BIỂU ĐỒ 3 BÊN & LƯU VỀ GOOGLE DRIVE 5TB
> Hiển thị bảng ma trận kết quả Clean + Degraded, tự động vẽ biểu đồ đối chiếu 3 bên (CVPR 2024 vs Plan A vs M4) và tự động đồng bộ sang Google Drive `logs/task_6.6/`.

In [ ]:
# ==============================================================================
# BƯỚC 5: BẢNG TỔNG HỢP ĐỐI CHIẾU 3 BÊN VÀ ĐỒNG BỘ GOOGLE DRIVE
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

res_clean_csv = os.path.join(OUTPUT_LOGS, 'final_posthoc_benchmark_matrix.csv')
res_deg_csv = os.path.join(OUTPUT_LOGS, 'final_locked_degraded_matrix.csv')
res_json = os.path.join(OUTPUT_LOGS, 'final_locked_benchmark_report.json')

if os.path.exists(res_clean_csv):
    df_clean = pd.read_csv(res_clean_csv)
    display(Markdown('### 📊 1. KẾT QUẢ KIỂM CHUẨN KHÓA SỔ TRÊN 18 TẬP TEST CLEAN:'))
    display(df_clean)

if os.path.exists(res_deg_csv):
    df_deg = pd.read_csv(res_deg_csv)
    display(Markdown('### 📉 2. KẾT QUẢ KIỂM CHUẨN KHÓA SỔ TRÊN 6 BIẾN THỂ DEGRADED:'))
    display(df_deg)

# Vẽ biểu đồ 3 bên nếu có script
plot_script = f'{REPO_DIR}/tools/plot_three_way_benchmark.py'
if os.path.exists(plot_script):
    try:
        subprocess.run([sys.executable, plot_script], check=True)
        chart_p = f'{REPO_DIR}/DATASET/logs/task_6.6_three_way_comparison.png'
        if os.path.exists(chart_p):
            shutil.copy2(chart_p, os.path.join(OUTPUT_LOGS, 'task_6.6_three_way_comparison.png'))
            print('[✓] Đã tạo biểu đồ đối chiếu 3 bên: task_6.6_three_way_comparison.png')
    except Exception as e:
        print(f'[!] Cảnh báo vẽ biểu đồ: {e}')

# 1. Đóng gói toàn bộ kết quả vào tệp zip
export_zip = '/content/task_6.6_final_results.zip'
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_LOGS):
        for fn in files:
            fp = os.path.join(root, fn)
            arcname = os.path.relpath(fp, OUTPUT_LOGS)
            zf.write(fp, arcname)

# 2. Tự động lưu trữ trực tiếp vào Google Drive 5TB
drive_out_dir = os.path.join(DRIVE_ROOT, "logs", "task_6.6")
os.makedirs(drive_out_dir, exist_ok=True)
for item in os.listdir(OUTPUT_LOGS):
    s_item = os.path.join(OUTPUT_LOGS, item)
    d_item = os.path.join(drive_out_dir, item)
    if os.path.isdir(s_item):
        shutil.copytree(s_item, d_item, dirs_exist_ok=True)
    else:
        shutil.copy2(s_item, d_item)

shutil.copy2(export_zip, os.path.join(drive_out_dir, "task_6.6_final_results.zip"))
print(f"[✓] ĐÃ TỰ ĐỘNG LƯU TRỮ TOÀN BỘ KẾT QUẢ VÀO GOOGLE DRIVE: {drive_out_dir}")

# 3. Kích hoạt tải trực tiếp về máy tính nếu muốn
try:
    from google.colab import files
    files.download(export_zip)
    print("[✓] Đã kích hoạt tải tệp task_6.6_final_results.zip về máy!")
except Exception:
    pass

print('\n🎉 HOÀN TẤT TOÀN DIỆN VÒNG ĐO ĐẠC TASK 6.6 TRÊN GOOGLE COLAB PRO!')

